# RunPod: the flat model (level 1), the complete cycle

Run the cells from the top, on an A100 pod with the network volume at `/workspace` and this repository cloned to `/workspace/AAC`. The dataset comes from the Hub repo that `runpod_dataset.ipynb` filled.

| step | what |
|---|---|
| 0 | configuration, helpers, Hugging Face login, the PyTorch environment, the dataset from the Hub |
| 1 | **smoke run** of the training on the real files (a few batches, then the probes), **timing** of a step, of the real data loader and of a planning solve |
| 2 | **F1**, the default flat model: training (detached, resumes after a pod restart), probes |
| 3 | the planning settings (**F2**, decided from F1), then **P1-P3** (F3, F4, F5) with the random baseline and videos |
| 4 | variants (**V1** context and rollout loss), the same cycle |
| 5 | runs and reports to the Hub |

Jobs are the ones of `experiments/experiments.yaml`, run by `experiments/run.py` (a job whose output exists is skipped). A step that finished is skipped when you run its cell again.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
from pathlib import Path

VOLUME = Path("/workspace")
REPO = VOLUME / "AAC"                      # the git clone
VENV = VOLUME / "venvs" / "hjepa"          # PyTorch environment, on the volume: a restarted pod reuses it
DATA = VOLUME / "data"                     # HJEPA_HOME: episode files, checkpoints, reports, logs

HF_REPO_ID = "your-user/craftax-classic"   # the dataset repo of runpod_dataset.ipynb (data/), runs go to runs/
HF_PRIVATE = True

SEEDS = "42"                               # comma-separated; headline results: "42,43,44"
MAX_EPOCHS = None                          # set from the timing in step 1 (the config's 10 are the Push-T placeholder)
TRAIN_OVERRIDES = ["trainer.devices=1"]    # Hydra overrides for every training run; one GPU: on several, DDP stops at the unused ViT mask_token
LOADER_BATCHES = 50                        # batches of the real loader timed in step 1
REPACK_PIXELS = False                      # step 1: rewrite the training file without compression if loading is the bottleneck

In [ ]:
# --- 0. Helpers, Hugging Face login ----------------------------------------------------------------------------------
import getpass, hashlib, json, os, re, shutil, subprocess, sys, time

os.chdir(VOLUME)   # a deleted and re-cloned repo leaves the kernel in a directory that no longer exists

PYTHON = VENV / "bin" / "python"
H_JEPA = REPO / "H-JEPA" / "h_jepa"
RUNNER = [PYTHON, REPO / "experiments" / "run.py"]
LOGS = DATA / "logs"
ENV = {
    # LD_LIBRARY_PATH is dropped: the RunPod image puts its own CUDA 12.8 there, next to the pip-installed CUDA libraries
    **{k: v for k, v in os.environ.items() if k != "LD_LIBRARY_PATH"},
    "PATH": f"{VENV / 'bin'}:{os.environ['PATH']}",   # `python` in the runner's commands is the venv's
    "HJEPA_HOME": str(DATA),
    "PYTHONPATH": f"{H_JEPA.parent}:{H_JEPA}",        # h_jepa modules import each other as top-level modules
    "HDF5_USE_FILE_LOCKING": "FALSE",                 # HDF5 can fail to lock files on a network volume
    "PYTHONUNBUFFERED": "1",
    "PYTHONFAULTHANDLER": "1",
}
LOGS.mkdir(parents=True, exist_ok=True)


def sh(cmd, cwd=None, env=None, check=True, quiet=False):
    """Run a command and stream its output into the cell; `quiet` shows the last lines only if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join((c.splitlines() or [""])[0][:100] + (" ..." if len(c.splitlines()) > 1 or len(c) > 100 else "") for c in cmd))
    proc = subprocess.Popen(cmd, env=env or ENV, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not quiet:
            print(line, end="")
    code = proc.wait()
    if code and quiet:
        print("".join(lines[-40:]), end="")
    if check and code:
        raise RuntimeError(f"{cmd[0]} exited with {code}")
    return code


PROCS = {}


def launch(name, cmd, cwd=REPO):
    """Start a command detached from the kernel (new session), log to LOGS/<name>.log, pid in LOGS/<name>.pid."""
    if running(name):
        return print(f"{name} is already running (pid {(LOGS / (name + '.pid')).read_text()})")
    with open(LOGS / f"{name}.log", "a") as log:
        PROCS[name] = subprocess.Popen([str(c) for c in cmd], env=ENV, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    (LOGS / f"{name}.pid").write_text(str(PROCS[name].pid))
    print(f"started {name}, pid {PROCS[name].pid}, log {LOGS / (name + '.log')}")


def running(name):
    if name in PROCS:                      # poll() also reaps a finished child
        return PROCS[name].poll() is None
    try:                                   # after a kernel restart the process is no longer our child
        os.kill(int((LOGS / f"{name}.pid").read_text()), 0)
        return True
    except (FileNotFoundError, ValueError, ProcessLookupError):
        return False


def progress(name, key, n=1):
    """The last `n` log lines that contain `key`; progress bars overwrite their line with \\r, so both split lines."""
    path = LOGS / f"{name}.log"
    lines = [l for l in re.split(r"[\r\n]", path.read_text(errors="replace")) if key in l] if path.exists() else []
    return [l.strip()[:200] for l in lines[-n:]] or ["(no progress line yet)"]


def wait(name, key, every=120):
    while running(name):
        time.sleep(every)
        print(time.strftime("%H:%M"), progress(name, key)[0])
    print(f"{name} finished")
    log = (LOGS / f"{name}.log").read_text(errors="replace")
    assert "failed with exit code" not in log, f"a job of {name} failed: read {LOGS / (name + '.log')}, then run its start cell again"


def run_dir(model, seed=SEEDS.split(",")[0]):
    return DATA / "ckpts" / "craftax" / model / f"seed{seed}"


def show(path, lines=60):
    path = Path(path)
    print("\n".join(path.read_text().splitlines()[:lines]) if path.exists() else f"({path} does not exist yet)")


sh([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=2"], env=os.environ)
from huggingface_hub import HfApi, hf_hub_download

api = HfApi(token=os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (write access): "))
print("logged in as", api.whoami()["name"])
api.create_repo(HF_REPO_ID, repo_type="dataset", private=HF_PRIVATE, exist_ok=True)

In [ ]:
# --- 0. The pod, the PyTorch environment (installed once) and the dataset ---------------------------------------------
assert REPO.exists(), f"clone the repository to {REPO}"
for command in (["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], ["nproc"], ["df", "-h", str(VOLUME)]):
    sh(command, env=os.environ, check=False)
sh(["git", "-C", REPO, "log", "-1", "--oneline"], env=os.environ)

files = [REPO / "environment" / "hjepa-requirements.txt"]
stamp = hashlib.sha256(b"".join(f.read_bytes() for f in files)).hexdigest()[:16]
marker = VENV / ".installed"
if not (marker.exists() and marker.read_text() == stamp):
    if not PYTHON.exists():
        sh([sys.executable, "-m", "venv", VENV], env=os.environ)
    sh([PYTHON, "-m", "pip", "install", "-q", "-U", "pip"])
    sh([PYTHON, "-m", "pip", "install", "-q", "-r", "environment/hjepa-requirements.txt"], cwd=REPO)
    marker.write_text(stamp)
cuda = subprocess.run([PYTHON, "-c", "import torch; print(torch.cuda.is_available(), torch.__version__)"], env=ENV, capture_output=True, text=True).stdout.strip()
assert cuda.startswith("True"), f"torch does not see the GPU ({cuda}): check nvidia-smi and the torch install in the venv"
print("torch sees the GPU:", cuda)

# the dataset of runpod_dataset.ipynb, from the Hub (skipped when the files are there)
for split in ("train", "val", "test"):
    if not (DATA / f"craftax_expert_{split}.h5").exists():
        hf_hub_download(HF_REPO_ID, f"data/craftax_expert_{split}.h5", repo_type="dataset", local_dir=VOLUME, token=api.token)
    print(split, f"{(DATA / f'craftax_expert_{split}.h5').stat().st_size / 2**30:.2f} GiB")

# the files must hold the executed actions (noop while asleep): relabel_dataset.ipynb wrote them to the Hub
RELABELED = "import h5py, sys; f = h5py.File(sys.argv[1], 'r'); print('relabel' in f.attrs and 'action_policy' in f)"
for split in ("train", "val", "test"):
    out = subprocess.run([PYTHON, "-c", RELABELED, DATA / f"craftax_expert_{split}.h5"], env=ENV, capture_output=True, text=True)
    assert out.stdout.strip() == "True", f"{split} is not relabeled: run notebooks/relabel_dataset.ipynb (or delete the local copy to download it again)"
print("the files hold the executed actions (action_policy: the policy's choice)")

## 1. Smoke run and timing

The **smoke run** is the real training command on the real files, a few batches and one validation pass, then the probes at the end (encoded latents, rollouts, place/make events, IDM per action). It proves the columns, the paths and the reports on this pod before the long run. The **timing** measures a training step, the real data loader (the pixels are stored gzip-compressed in chunks of 100 frames, so loading can be the bottleneck) and a planning solve, and gives the GPU-hours for the configured epochs. Set `MAX_EPOCHS` from it.

In [ ]:
# --- 1. Smoke run on the real files --------------------------------------------------------------------------------------
SMOKE = "flat_smoke"
shutil.rmtree(run_dir(SMOKE), ignore_errors=True)
t0 = time.time()
sh([PYTHON, "main_hjepa.py", "--config-name", "craftax_hjepa", "num_levels=1", f"output_model_name={SMOKE}", "trainer.max_epochs=1",
    "+trainer.limit_train_batches=20", "+trainer.limit_val_batches=2", "health.probe_clips=64", *TRAIN_OVERRIDES], cwd=H_JEPA, quiet=True)
out = run_dir(SMOKE)
assert (out / f"{SMOKE}_object.ckpt").exists(), "the smoke run wrote no model"
assert (out / "health" / "report.md").exists() and (out / "health" / "probe_report.md").exists(), "the smoke run wrote no reports"
print(f"smoke run ok in {time.time() - t0:.0f} s: {out}")
show(out / "health" / "probe_report.md", 25)

In [ ]:
# --- 1. Timing: a training step, the real loader, a planning solve ---------------------------------------------------------
sh([PYTHON, "time_run.py", "--config-name", "craftax_hjepa", "--eval-config-name", "craftax_plan", "--loader-batches", LOADER_BATCHES,
    "num_levels=1", *TRAIN_OVERRIDES], cwd=H_JEPA)

In [ ]:
# --- 1. Optional: the training file without compression (only if the timing says loading is the bottleneck) ----------------
REPACK = """
import os, sys, h5py, numpy as np
path = sys.argv[1]
tmp = path + ".repack"
with h5py.File(path, "r") as src, h5py.File(tmp, "w") as dst:
    for key, value in src.attrs.items():
        dst.attrs[key] = value
    for name, ds in src.items():
        out = dst.create_dataset(name, shape=ds.shape, dtype=ds.dtype)   # contiguous, no compression
        step = 20_000
        for a in range(0, ds.shape[0], step):
            out[a : a + step] = ds[a : a + step]
        for key, value in ds.attrs.items():
            out.attrs[key] = value
    rng = np.random.default_rng(0)
    for name in src:
        rows = np.sort(rng.choice(src[name].shape[0], size=min(200, src[name].shape[0]), replace=False))
        assert np.array_equal(src[name][rows], dst[name][rows], equal_nan=src[name].dtype.kind == "f"), name
os.replace(tmp, path)
print("repacked", path)
"""
train_file = DATA / "craftax_expert_train.h5"
if REPACK_PIXELS:
    sh(["df", "-h", str(DATA)], env=os.environ)   # needs about 25 GB free
    sh([PYTHON, "-c", REPACK, train_file])
    sh([PYTHON, "time_run.py", "--config-name", "craftax_hjepa", "--steps", "3", "--loader-batches", LOADER_BATCHES, "num_levels=1", *TRAIN_OVERRIDES], cwd=H_JEPA)
else:
    print("REPACK_PIXELS is False: the training file stays as it is")

## 2. F1, the default flat model

`craftax_hjepa num_levels=1`: history 3, `rollout_n` 1, the CLS latent, IDM on. The job trains, then probes (`probe_eval.py`, 512 clips) and writes the comparison report. If the pod restarts, run the start cell again: the training resumes from `lightning_resume/last.ckpt`, the finished jobs are skipped.

In [ ]:
# --- 2. Start (or resume) F1, detached ---------------------------------------------------------------------------------------
assert MAX_EPOCHS, "set MAX_EPOCHS in the configuration cell from the timing"
OVERRIDES = [f"trainer.max_epochs={MAX_EPOCHS}", *TRAIN_OVERRIDES]
sh([*RUNNER, "plan", "F1", "--seeds", SEEDS, "--override", *OVERRIDES])
launch("F1", [*RUNNER, "run", "F1", "--mode", "local", "--seeds", SEEDS, "--override", *OVERRIDES])

In [ ]:
# --- 2. Monitor F1 (run as often as you like) ----------------------------------------------------------------------------------
print("F1", "RUNNING" if running("F1") else "not running")
print("\n".join(progress("F1", "Epoch", 2)))
print("\n".join(progress("F1", "[health]", 5)))
for p in sorted((run_dir("flat") / "health").glob("epoch_*.json"))[-1:]:
    print("last epoch report:", p.name)

In [ ]:
# --- 2. Wait for F1, then the training and probe reports ------------------------------------------------------------------------
wait("F1", "Epoch")
show(run_dir("flat") / "health" / "report.md", 40)
show(run_dir("flat") / "health" / "probe_report.md", 120)

## 3. Planning: the settings (F2), then P1-P3

**F2 is not designed yet**: the goal offset, the planning horizon and the CEM settings are chosen from the F1 probes (the readable rollout horizon) and a few planning runs on the validation worlds, then frozen for every variant. Put the chosen values into `experiments/experiments.yaml` (task sets A, B, C) and `H-JEPA/h_jepa/config/eval/craftax_plan.yaml`, commit, and pull here before starting P1-P3.

P1 = F3 (fixed-offset goals, task set A), P2 = F4 (oracle subgoals, task set C), P3 = F5 (final goal only, task set B), each with the random baseline on the same tasks and GIFs of the first 8 tasks (`videos/`).

In [ ]:
# --- 3. Pull the frozen planning settings, then start P1-P3 one after the other, detached -------------------------------------------
sh(["git", "-C", REPO, "pull", "--ff-only"], env=os.environ)
sh(["git", "-C", REPO, "log", "-1", "--oneline"], env=os.environ)
CHAIN = """
import subprocess, sys
runner, seeds = sys.argv[1:3]
for exp in sys.argv[3:]:
    print("=====", exp, flush=True)
    code = subprocess.call([sys.executable, runner, "run", exp, "--mode", "local", "--seeds", seeds])
    if code:
        sys.exit(f"{exp} failed with exit code {code}")
"""
launch("planning", [PYTHON, "-c", CHAIN, RUNNER[1], SEEDS, "F3", "F4", "F5"])

In [ ]:
# --- 3. Monitor and read P1-P3 (run as often as you like) ---------------------------------------------------------------------------
print("planning", "RUNNING" if running("planning") else "not running")
print("\n".join(progress("planning", "=====", 1) + progress("planning", "success_rate", 3)))
for metrics in sorted((DATA / "ckpts" / "craftax").glob("*/**/eval_*/metrics.yaml")):
    rate = next((l for l in metrics.read_text().splitlines() if l.startswith("success_rate")), "")
    videos = len(list((metrics.parent / "videos").glob("*.gif")))
    print(f"{metrics.parent.relative_to(DATA)}: {rate}, {videos} videos")
for report in sorted((DATA / "reports").glob("F[345]_*.md")):
    print("\n=====", report.name)
    show(report, 30)

## 4. Variants

**V1**: context (history 1) and rollout loss (`rollout_n` 4), each trained, probed and planned like F1 with the frozen settings (placeholders for the values in `experiments.yaml`). The SIGReg and IDM coefficients, delta prediction and patch tokens are added to `experiments.yaml` when they are run (README, "Current phase").

In [ ]:
# --- 4. Start V1, detached -----------------------------------------------------------------------------------------------------------
assert MAX_EPOCHS, "set MAX_EPOCHS"
OVERRIDES = [f"trainer.max_epochs={MAX_EPOCHS}", *TRAIN_OVERRIDES]
sh([*RUNNER, "plan", "V1", "--seeds", SEEDS, "--override", *OVERRIDES])
launch("V1", [*RUNNER, "run", "V1", "--mode", "local", "--seeds", SEEDS, "--override", *OVERRIDES])

In [ ]:
# --- 4. Monitor V1 ---------------------------------------------------------------------------------------------------------------------
print("V1", "RUNNING" if running("V1") else "not running")
print("\n".join(progress("V1", "Epoch", 2) + progress("V1", "success_rate", 3)))
for report in sorted((DATA / "reports").glob("V1_*.md")):
    print("\n=====", report.name)
    show(report, 30)

## 5. Upload

The runs (final models, configs, health and probe reports, planning results with their videos) and the comparison reports go to `runs/` and `reports/` of the same Hub repo. Left out: the resumable training state (`lightning_resume/`), the per-epoch models and the smoke run.

In [ ]:
# --- 5. Runs and reports to the Hub --------------------------------------------------------------------------------------------------------
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(DATA / "ckpts" / "craftax"), path_in_repo="runs",
                  ignore_patterns=["**/lightning_resume/**", "*_epoch_*_object.ckpt", "flat_smoke/**", "**/hydra/**", "**/spt/**"])
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(DATA / "reports"), path_in_repo="reports",
                  allow_patterns=["F*.md", "V*.md"])
print(f"done: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/runs")